# Generate your own embroidery stitch textures

Companion notebook for [*Digitally Reintegrating Losses in Heritage Embroidery*](https://geojenks.github.io/digitally-reintegrating-losses/) (GCH 2026).

It reintegrates losses in an embroidery photograph: each masked loss region is seeded with a **procedural texture init** (knot bumps / parallel satin threads / coiled purl) and restyled by **SDXL + a stitch-structure LoRA** trained on 17th-century English embroideries.

**Ways to run this:**
- **Free**: Colab's free T4 GPU runs the SDXL route as-is. Just run the cells top to bottom.
- **Paid**: with Colab Pro (A100/L4) — or your own billed Hugging Face account for the gated FLUX.1-dev weights — you can switch to the stronger FLUX route used for the paper's final figures (see the last cell).
- **Local, free**: any CUDA GPU with ~12 GB VRAM runs the SDXL route; ~24 GB runs FLUX. Clone the repo and follow its README — no notebook needed.


In [ ]:
#@title 1 — Check the GPU
!nvidia-smi -L

In [ ]:
#@title 2 — Get the code and dependencies (~2 min)
!git clone -q https://github.com/geojenks/digitally-reintegrating-losses
%cd digitally-reintegrating-losses
!pip -q install diffusers transformers accelerate safetensors sentencepiece protobuf opencv-python scipy

In [ ]:
#@title 3 — Download the SDXL stitch LoRAs (85 MB each)
import os, urllib.request
REL = "https://github.com/geojenks/digitally-reintegrating-losses/releases/latest/download"
for run in ["french_knot_stitch_sdxl_lora_v1",
            "satin_stitch_sdxl_lora_v1",
            "silk_purl_sdxl_lora_v1"]:
    os.makedirs(f"output/{run}", exist_ok=True)
    dst = f"output/{run}/{run}.safetensors"
    if not os.path.exists(dst):
        print("downloading", run)
        urllib.request.urlretrieve(f"{REL}/{run}.safetensors", dst)
print("LoRAs ready")

## Choose an image

By default the next cell uses the bundled **castle demonstration piece** and its three loss masks. To use **your own image**: run the upload cell instead — you need the image plus one black-and-white mask per stitch type (white = the loss to fill), named `<image>__satin.png`, `<image>__french_knot.png`, `<image>__silk_purl.png` (any subset is fine). The project page's picker demo can draw these masks for you (magic wand / box tools) and export them.


In [ ]:
#@title 4a — Use the bundled castle sample
IMAGE = "data/masks/castle.png"
MASKS = "data/masks"
print("using", IMAGE)

In [ ]:
#@title 4b — (optional) Upload your own image + masks
from google.colab import files
import os
os.makedirs("uploads", exist_ok=True)
up = files.upload()
for name, blob in up.items():
    open(f"uploads/{name}", "wb").write(blob)
imgs = [n for n in up if "__" not in n]
assert len(imgs) == 1, "upload exactly one image plus its __<stitch>.png masks"
IMAGE = f"uploads/{imgs[0]}"
MASKS = "uploads"
print("using", IMAGE, "with masks:", [n for n in up if "__" in n])

In [ ]:
#@title 5 — Procedural init parameters
knot_radius = 16        #@param {type:"slider", min:6, max:28, step:2}
satin_thread = 0        #@param {type:"slider", min:0, max:14, step:1}
satin_angle_step = 0    #@param {type:"slider", min:0, max:45, step:5}
purl_coil = 0           #@param {type:"slider", min:0, max:18, step:1}
denoise = 0.6           #@param {type:"slider", min:0.3, max:0.9, step:0.05}
seed = 1600             #@param {type:"integer"}
extra = []
if satin_thread: extra += ["--proc_satin_thread", str(satin_thread)]
if satin_angle_step: extra += ["--proc_satin_angle_step", str(satin_angle_step)]
if purl_coil: extra += ["--proc_coil", str(purl_coil)]
print("ready")

In [ ]:
#@title 6 — Reintegrate (SDXL; ~2-5 min per stitch on a T4)
import subprocess, sys
cmd = [sys.executable, "pipeline/staged_reintegrate.py",
       "--image", IMAGE, "--masks", MASKS,
       "--model", "sdxl_base", "--lora_variant", "v1",
       "--size", "1024", "--tries", "1",
       "--tex_proc", "--per_region", "--region_max_up", "2",
       "--proc_knot_r", str(knot_radius),
       "--brim", "10", "--sib_feather", "3",
       "--denoise", str(denoise), "--seed", str(seed),
       "--out", "staged_out"] + extra
print(" ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
#@title 7 — Show the result
import glob
from IPython.display import Image as I, display
finals = sorted(glob.glob("staged_out/**/*final*.png", recursive=True)) \
      or sorted(glob.glob("staged_out/**/*.png", recursive=True))
for f in finals[-3:]:
    print(f); display(I(f, width=700))

## Going further

- **FLUX route (paper quality)**: needs ~24 GB VRAM (Colab Pro A100, or locally). Accept the licence for [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev), download `flux1-dev.safetensors`, set `FLUX_DEV_SAFETENSORS` to its path, download the `*_flux1_lora_trigonly_v2` LoRAs from the release, and rerun cell 6 with `--model flux_base --lora_variant trigonly_v2`.
- **Per-region variants + picker**: add `--region_variants 8` and open `demo/region_picker.html` on the outputs to compose your favourite fill per region.
- **Train your own stitch LoRA**: you are limited to the three stitch types here unless you train your own — see `training/` in the repo for the exact ai-toolkit configs and `data/stitches/` for the captioning style.
